# Copia pública del 22 de octubre de 2026

Código de investigación; revisar entradas en `../README.md`. Las salidas guardadas se retiraron. Las etapas 01–06 no se reejecutaron para esta publicación. Para la síntesis numérica utilizada en la memoria, ejecutar `07_Sintesis_resultados_20261022.ipynb`. Las utilidades interactivas son opcionales de revisión y no una dependencia de la síntesis científica.

In [ ]:
from pathlib import Path
import sys
BASE = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'Configuracion.json').is_file()), None)
if BASE is None:
    raise FileNotFoundError('Iniciar Jupyter en notebooks/actual_20261022 o su subcarpeta de notebooks.')
sys.path.insert(0, str(BASE))
import publicacion_config


# 03 · K, U y τ: calibraciones oficiales y alternativas separadas
**Versión principal consolidada · revisión del 19 de octubre de 2026.**

La revisión manual guardada es la autoridad para B, MIS y rol. El flujo recalcula H, todos los interceptos y K; retiene primero / intermedio real más cercano a distancia media / último por B/MIS. Los productos principales usan sólo los B oficiales. Comparación y prueba tienen tablas, figuras y τ en carpetas y secciones propias.

**Selección vigente:** Malhueco A01–5e; Pinotalca N06–5e; Reloca N04–5e; Chanco N02–5e y A04–5c; Rahue N07–5e; Curanilahue A08–5e y A07–5c. N05–5c permanece en prueba; su asignación anterior 5e queda como comparación histórica.

Los ocho casos oficiales tienen 24 etiquetas de P_T y 18 posiciones distintas al contarlas por caso. Las coincidencias no son mediciones independientes. No se promedian K entre MIS ni se adopta un K único. H es el DEM en la terraza más cercana del mismo MIS; B sólo fija χ(B).

Los τ de este notebook usan K constante por escenario; el experimento de Reloca con dos K se mantiene separado. Los textos de metodología, resultados y sensibilidad permanecen en la carpeta de revisión.

**Cronología vigente, 20/10/2026:** se emplean las edades asignadas a los highstands en la Tabla 2 publicada de Jara-Muñoz et al. (2015), p. 179: 5e = 125 ± 5 ka y 5c = 106 ± 5 ka. Los datos internos mantienen su función geométrica y de correlación; las edades numéricas son las publicadas.

## 1. Configuración única y revisión manual
Ejecutar desde un kernel limpio produce una carpeta nueva. Los filtros de visualización no cambian las tablas oficiales.

`REVISION_B` contiene decisiones y resultados numéricos previamente revisados. El recálculo debe reproducirlos antes de actualizar la última corrida. B nuevos o coordenadas distintas requieren revisión: no se adoptan automáticamente. El CSV del visor se puede reconstruir con la opción existente, pero debe concordar con la revisión para entrar en este cierre.

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import sys,json,importlib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display,Image,HTML,Markdown
BASE=Path(str(BASE / 'inputs_external' / 'Jupyter_Memoria'))
APOYO=BASE/'Codigo_consolidado_20261019'
if str(APOYO) not in sys.path:sys.path.insert(0,str(APOYO))
import revision_b_base as rb
import escenarios_k_manuales as km
import calibracion_tau as ct
import calculo_unificado as cu
import visor_rios_swath as vr
import swath_b_3d as sb
import integracion_K_oficial as io
import cronologia_jm2015 as pub
for m in [rb,km,ct,cu,vr,sb,io]:importlib.reload(m)
CONFIG=json.loads((BASE/'Configuracion.json').read_text(encoding='utf-8-sig'))
DEM=BASE/CONFIG['dem'];KP_MANUALES=BASE/CONFIG['kp_manual_csv']
THETA=0.40;A0=1e6;N=1
assert CONFIG['theta']==THETA and CONFIG['A0_m2']==A0 and CONFIG['n']==N
REVISION_B=BASE/'Notebooks_principales_20261019/Revision_K_por_rio_20261019'
RIOS_CALCULAR=None
CSV_PUNTOS_B=Path(str(BASE / 'inputs_external' / 'Puntos_B_todos_los_rios.csv'))
RECONSTRUIR_B_DESDE_CSV=False
REGENERAR_VISOR_SWATH=False
ANCHO_SWATH_M=1000.0;LARGO_SWATH_M=2000.0;EXAGERACION_VERTICAL=1.0
MOSTRAR_COLORES_PENDIENTE=True
MOSTRAR={'swath':True,'figura_por_rio':True,'dispersion_K':True,'tau_perfiles':True,
         'K_tau_distancia':True,'norte_sur':True,'tabla_resumen':True,'comparacion_prueba':True}
MOSTRAR_K={'MIS5e':True,'MIS5c':True,'B_anteriores':True,'B_nuevos':True,'ascendentes':True,'descendentes':True}
SERIES={'propio':True,'K_JM':True,'K_pendiente':False,'Grant':False,'U_aparente':True,
        'U_terraza':True,'JM_U_publicado':True,'escenarios':True,'lineas':True,'errores_publicados':True}
MOSTRAR_TODAS_CURVAS_TAU=True
NIVEL_MAR={'5e':{'nivel_m':5.,'error_m':5.},'5c':{'nivel_m':-20.,'error_m':5.}}
RUN=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%fZ')
SALIDA=BASE/'Productos/03_K_U_tau'/RUN
SALIDA.mkdir(parents=True,exist_ok=False)
VISOR=REVISION_B/'Visor_swath_completo_B_20261019/Visor_swath_B_completo_20261019.html'
plt.rcParams.update({'font.size':10,'figure.facecolor':'white','axes.facecolor':'white'})

def enlace(nombre,p):
    from urllib.parse import quote
    from html import escape
    href='/files/'+quote(Path(p).relative_to(Path(str(BASE))).as_posix(),safe='/')
    display(HTML('<p><a target="_blank" href="'+escape(href,quote=True)+'"><b>'+escape(nombre)+'</b></a></p>'))
print('Resultados:',SALIDA)
print('Revisión manual:',REVISION_B)


## 2. Empezar por el perfil swath interactivo
El selector **Río / MIS** cambia de caso sin abrir otro HTML. Ctrl + clic conserva un candidato de B; exportar el CSV y actualizarlo en la sección 1 permite incorporarlo al cálculo posterior.

**Colorear pendiente** activa azul-rojo; desactivarlo deja el relieve en gris sin alterar cotas, puntos o áreas. Ancho, largo y exageración vertical se ajustan en el propio visor. Las terrazas 5e/5c tienen controles independientes. Los puntos del navegador dependen del origen: importar el CSV de respaldo si se cambia de navegador o ruta.

El visor inicial ya existe y no exige recalcular. Si se añadieron B al CSV, se puede regenerar más abajo. Una marca nueva no se considera una paleodesembocadura demostrada por el mero hecho de haberla clicado.

In [ ]:
if MOSTRAR['swath']:
    if not VISOR.exists():raise FileNotFoundError(VISOR)
    enlace('ABRIR SWATH COMPLETO: B oficiales, comparación y prueba',VISOR)
print('El visor conserva Ctrl+clic, CSV, pendiente, terrazas y tamaño editable; no cambia la selección oficial.')

## 3. Geometría y decisiones oficiales
Se conserva la geometría verificada y los 18 perfiles de contexto. Se aplica la revisión por ID de B, comprobando XY, χ(B), distancia y cota. La reclasificación N05–5c procede de la decisión manual, no de su etiqueta antigua en el CSV. Su caso 5e permanece separado como histórico.

Las alternativas de B conservan sus coordenadas. El rol oficial no significa que se haya escogido un K único o que la cronología haya sido validada independientemente.

In [ ]:
casos_entrada,perfiles,nombres,control_B,origen_geometria=cu.geometria(BASE,APOYO,SALIDA,RECONSTRUIR_B_DESDE_CSV,CSV_PUNTOS_B)
casos_entrada.to_csv(SALIDA/'B_entrada_sin_revision.csv',index=False,encoding='utf-8-sig')
casos_todos,decisiones=io.aplicar_decisiones(casos_entrada,REVISION_B)
if RIOS_CALCULAR is not None:casos_todos=casos_todos[casos_todos.rid.isin(RIOS_CALCULAR)].copy()
casos=casos_todos[casos_todos.rol_resultado.eq('oficial')].copy()
catalogo=pd.read_csv(APOYO/'Fuentes/terrazas_170_UTM18S_con_edad_referencia_Jara_Munoz_2015.csv',float_precision='round_trip')
origen_geometria+='; revisión manual B/MIS/rol aplicada y verificada'
print(origen_geometria)
display(casos_todos[['rid','id_B_corto','mis','rol_resultado','caso_B']])
print('Oficiales:',len(casos),'casos B/MIS; todos los roles:',len(casos_todos))

### Regeneración opcional del visor
La vista predeterminada es el visor completo ya revisado. La regeneración opcional crea otro archivo sin borrar el anterior. Incluye los casos revisados de todos los roles; las posiciones nuevas requieren concordar con la revisión antes de incorporarse al cálculo.

In [ ]:
if REGENERAR_VISOR_SWATH:
    elementos=[]
    for _,b in casos_todos[casos_todos.B_computable].iterrows():
        p=perfiles[int(b.rid)]
        banda=sb.preparar_banda(DEM,b.x_B,b.y_B,semiancho_ew_m=ANCHO_SWATH_M/2,semilargo_ns_m=LARGO_SWATH_M/2,paso_m=None)
        fig=sb.graficar_banda(banda,nombres[int(b.rid)],b.caso_B,perfil_rio=p,puntos_JM=catalogo,etiquetar_terrazas=True,exageracion_vertical=EXAGERACION_VERTICAL)
        contexto,pts=sb.mapa_contexto_terrazas(banda,catalogo,perfil_rio=p,semilado_m=3000,etiquetar=True)
        elementos.append({'banda':banda,'figura':fig,'contexto':contexto,'terrazas_contexto':pts,'perfil_rio':p,
            'semilado_contexto_m':3000,'caso_B':b.caso_B,'rid':int(b.rid),'rio':nombres[int(b.rid)],'mis':b.mis,'exageracion_vertical':EXAGERACION_VERTICAL})
    VISOR=vr.exportar_visor(elementos,SALIDA/'Swath_3D')
    enlace('ABRIR VISOR REGENERADO',VISOR)

## 4. Recalcular K y retener tres P_T por caso
H se obtiene del DEM en la terraza J-M más cercana del mismo MIS. Se recalculan todas las intersecciones aguas arriba de B y sus K. De cada caso se retienen primero, último e intercepto real más cercano a la distancia media entre ambos. No se promedia K para construir el escenario intermedio.

$$K_j=\frac{\chi(P_{T,j})-\chi(B)}{A_0^{0,40}(1000T_{ka})}.$$

El resultado principal usa sólo B oficiales. Los cruces completos y las alternativas se conservan en archivos separados. Cada etiqueta tiene un ID único, incluso cuando varias etiquetas representan la misma posición P_T. T=125±5 ka para 5e y 105±5 ka para 5c conserva el catálogo adoptado; la discrepancia 105/106 ka y su efecto <1 % están documentados en el cierre. El ±5 ka se trata como rango cronológico compartido por MIS, no como error independiente de datación de cada punto.

In [ ]:
referencias_todas,resultados_todos,control_PT_todos=cu.calcular_K(casos_todos,perfiles,nombres,catalogo,DEM)
seleccion_todos=io.seleccionar_tres(resultados_todos)
control_revision=io.verificar_revision(seleccion_todos,REVISION_B,perfiles,SALIDA/'Controles')
io.exportar_roles(casos_todos,referencias_todas,resultados_todos,seleccion_todos,REVISION_B,SALIDA)
referencias=referencias_todas[referencias_todas.rol_resultado.eq('oficial')].copy()
resultados=seleccion_todos[seleccion_todos.rol_resultado.eq('oficial')].copy()
control_PT=control_PT_todos[control_PT_todos.caso_altura.isin(referencias.caso_altura)]
control_PT.to_csv(SALIDA/'Control_PT.csv',index=False,encoding='utf-8-sig')
control_PT_todos.to_csv(SALIDA/'Archivo_escenarios/Control_PT_todos_roles.csv',index=False,encoding='utf-8-sig')
print('Oficiales:',len(referencias),'casos y',len(resultados),'etiquetas de P_T.')
print('Cruces oficiales completos:',int(resultados_todos.rol_resultado.eq('oficial').sum()))
print('Reproducción de la revisión:',control_revision)
assert resultados.id_medicion.is_unique and resultados.rol_resultado.eq('oficial').all()

## 5. Figuras oficiales por río
Mapa, χ, perfil longitudinal y K se alimentan sólo con los ocho casos oficiales y sus tres etiquetas P_T. Las alternativas no se mezclan en estos paneles. Las capas geológicas no dibujadas permanecen vacías. Los B oficiales conservan sus tres escenarios de calibración.

In [ ]:
if MOSTRAR['figura_por_rio']:
    capas=cu.capas_revision(BASE,APOYO,catalogo)
    referencias_fig,resultados_fig=cu.adaptar_figura(referencias,resultados)
    rb.figuras_por_rio(referencias_fig,resultados_fig,perfiles,nombres,capas,DEM,SALIDA/'Figuras_por_rio')
    for f in sorted((SALIDA/'Figuras_por_rio').glob('*.png')):display(Image(filename=str(f),width=1250))

## 6. Dispersión de K e histogramas por río/MIS
Los MIS se grafican por separado, sin promedio de K. Se conservan los ID de escenario usados por τ y las tablas finales. Los histogramas cuentan etiquetas de escenario; las coincidencias geométricas no son mediciones independientes. La referencia de Grant sigue siendo una comparación externa, sin seleccionar K automáticamente.

In [ ]:
mediciones,figuras_K,html_K=io.figuras_K(resultados,SALIDA/'K_Grant',MOSTRAR_K,ct.limites_grant())
if MOSTRAR['dispersion_K']:
    for f in figuras_K:display(Image(filename=str(f),width=1350))
enlace('K oficiales por MIS: dispersión, histogramas y Grant',html_K)

## 7. τ con K, B y perfiles oficiales
$$\tau_B(x)=\frac{\chi(x)-\chi(B)}{K A_0^{0.40}}.$$
Se divide por 1000 para expresar ka y se calcula únicamente aguas arriba de B. Los KP se recuperan del CSV manual por XY y se interpolan sobre este mismo perfil; no se reutiliza un χ de otra versión. Chanco conserva perfiles pero no recibe un KP inventado.

Se dibuja cada escenario y su rango por B. **τ(P_T)=T por construcción**, útil como control de unidades, no como validación independiente. Extender τ al interior supone K constante y área actual; no fecha de forma independiente las fallas.

Estos τ corresponden a K constante por escenario. No se ejecuta aquí la propagación por dos litologías con dos K.

In [ ]:
tau_fig,tau_KP,tau_rangos,control_tau=ct.calcular_tau(resultados[resultados.K_m02_anio.gt(0)],perfiles,KP_MANUALES,SALIDA/'Tau',A0,THETA,MOSTRAR_TODAS_CURVAS_TAU)
if MOSTRAR['tau_perfiles']:
    for f in tau_fig:display(Image(filename=str(f),width=1350))
print('Error máximo de τ(PT)−T:',control_tau['max_error_tau_PT_menos_T_ka'],'ka')

## 8. Tasas de alzamiento y tasas aparentes: no confundir estimadores
**U de terraza:** `(H_publicada − nivel_mar_MIS)/T_MIS`, con errores publicados de H y T y la incertidumbre del nivel marino ya adoptada. La altura publicada se utiliza aquí para compararla con el nivel marino; el datum absoluto del DEM sigue sin confirmar, por eso su cota no sustituye silenciosamente a H publicada en esta tasa.

**U aparente experimental:** pendiente OLS del mismo tramo B–P_T en z–τ, con intercepto libre y al menos cinco nodos reales. Como τ es una transformación lineal de χ,
$$U_{aparente}=\frac{dz}{d\tau}=1000\,K\,k_{sn}\quad\text{(mm/año, si τ se ajusta en ka).}$$
Estas dos expresiones son equivalentes y **no son dos estimaciones independientes**. La tasa representa la capacidad de incisión del modelo; sólo se identifica con U bajo equilibrio. Con transitorios y variaciones del nivel de base no reconstruye por sí sola una historia de uplift.

**J-M:** tasas publicadas en la latitud real del marcador. **K derivado de U J-M:** calculado aquí usando ese U y el ksn del mismo B–P_T; no es un K publicado por J-M. **Grant:** U inferido = K_Grant×ksn y τ inferido con el mismo B; son transferencias experimentales, no sus tasas de erosión medidas.

Referencia: [Royden y Perron (2013)](https://agupubs.onlinelibrary.wiley.com/doi/10.1002/jgrf.20031).

In [ ]:
comparaciones=cu.tasas_y_comparaciones(resultados,perfiles,catalogo,tau_KP,NIVEL_MAR)
comparaciones.to_csv(SALIDA/'K_tau_U_todos_los_escenarios.csv',index=False,encoding='utf-8-sig')
valid=comparaciones.U_aparente_tau_mm_anio.notna()
assert np.allclose(comparaciones.loc[valid,'U_aparente_tau_mm_anio'],comparaciones.loc[valid,'U_aparente_Kksn_mm_anio'],rtol=1e-10)
print('Tasas aparentes B–PT calculadas:',int(valid.sum()))

## 9. K y τ en el perfil y de norte a sur
En cada escenario K se supone constante a lo largo del cauce; el gráfico no inventa una variación espacial de K. τ sí aumenta aguas arriba. Las comparaciones de latitud conservan todos los valores oficiales y unen medianas descriptivas por río/MIS, con sus rangos de escenarios.

La latitud de cada río corresponde al outlet de análisis original. Las tasas J-M conservan la latitud de su marcador. Carranza y Pelluhue se indican en la latitud del extremo occidental de sus trazas. Los True/False de `SERIES` permiten ocultar Grant, K derivado de U J-M, U de terraza, U aparente, errores y líneas.

In [ ]:
if MOSTRAR['K_tau_distancia']:
    for f in cu.figuras_perfil_K_tau(comparaciones,perfiles,SALIDA/'Perfiles_K_tau'):display(Image(filename=str(f),width=1200))
fallas=gpd.read_file(BASE/CONFIG['chaf']).to_crs(32718)
figuras_NS,html_NS=cu.comparacion_norte_sur(comparaciones,catalogo,fallas,SALIDA/'Norte_sur',SERIES)
if MOSTRAR['norte_sur']:
    for f in figuras_NS:display(Image(filename=str(f),width=1250))
enlace('K, τ y U de norte a sur: mostrar/ocultar series',html_NS)

## 10. Comparación y prueba: productos separados
Los B no oficiales y N05–5c permanecen accesibles aquí. Cada rol tiene sus propios CSV, figuras, curvas τ y tasas aparentes. Las referencias históricas no alimentan los resultados oficiales. Los mismos supuestos de K constante y cronología condicionada se mantienen en esta sección.

In [ ]:
productos_otros={}
for rol,carpeta in [('comparacion','Comparacion'),('prueba','Prueba')]:
    datos_rol=seleccion_todos[seleccion_todos.rol_resultado.eq(rol)].copy()
    if datos_rol.empty:continue
    ruta=SALIDA/carpeta
    refs_rol=referencias_todas[referencias_todas.rol_resultado.eq(rol)].copy()
    _,figs_rol,html_rol=io.figuras_K(datos_rol,ruta/'K_Grant',MOSTRAR_K,ct.limites_grant())
    figs_tau_rol,kp_rol,rangos_rol,control_tau_rol=ct.calcular_tau(datos_rol,perfiles,KP_MANUALES,ruta/'Tau',A0,THETA,MOSTRAR_TODAS_CURVAS_TAU)
    tasas_rol=cu.tasas_y_comparaciones(datos_rol,perfiles,catalogo,kp_rol,NIVEL_MAR)
    tasas_rol.to_csv(ruta/'K_tau_U_tres_PT.csv',index=False,encoding='utf-8-sig')
    assert tasas_rol.rol_resultado.eq(rol).all()
    productos_otros[rol]={'html':html_rol,'control_tau':control_tau_rol,'n_escenarios':len(datos_rol)}
    if MOSTRAR['comparacion_prueba']:
        display(Markdown('### '+carpeta))
        display(datos_rol[['rio','id_B_corto','mis','escenario_PT','terraza_id','H_m','T_ka','K_m02_anio']])
        for f in figs_rol:display(Image(filename=str(f),width=1200))
        enlace('Abrir K: '+carpeta,html_rol)
print('Oficiales, comparación y prueba se exportaron por separado.')

## 11. Exportación final y verificación de la corrida
La comprobación principal reproduce la revisión oficial vigente, no la antigua selección de la v8. Se verifican B/MIS/rol, H, T, posiciones P_T, K, ID únicos y el control τ(P_T)=T por construcción. Los productos principales contienen sólo oficiales; las alternativas conservan sus propios archivos.

El nombre compatible `K_todos_los_cruces.csv` contiene las mismas 24 etiquetas oficiales utilizadas por `Tau/`. `K_todos_interceptos.csv` conserva todos los cruces oficiales y `Archivo_escenarios/` incluye los cruces y escenarios de todos los roles. La última corrida se actualiza al superar los controles.

In [ ]:
tabla=cu.resumen_png(comparaciones,SALIDA/'Tablas')
if MOSTRAR['tabla_resumen']:display(Image(filename=str(tabla),width=1500))
errores=[]
for _,v in resultados.iterrows():
    p=perfiles[int(v.rid)]
    errores.append(abs(np.interp(v.s_PT_m,p.distancia_m,p.elevacion_m)-v.H_m))
assert max(errores)<1e-6
assert resultados.id_medicion.is_unique and resultados.escenario_K.is_unique
assert len(resultados)==3*len(casos) and resultados.groupby('caso_B').size().eq(3).all()
assert resultados.rol_resultado.eq('oficial').all() and comparaciones.rol_resultado.eq('oficial').all()
assert set(tau_KP.id_medicion).issubset(set(resultados.id_medicion))
if RIOS_CALCULAR is None:
    assert len(casos)==8 and len(resultados)==24
    assert len(seleccion_todos)==51
    assert set(resultados.id_B_corto)=={'A01','N06','N04','N02','A04','N07','A08','A07'}
assert control_tau['max_error_tau_PT_menos_T_ka']<1e-8
from red_fallas import sha
sources=[APOYO/'cronologia_jm2015.py',APOYO/'integracion_K_oficial.py',APOYO/'calculo_unificado.py',APOYO/'calibracion_tau.py',
         APOYO/'Fuentes/Puntos_B_todos_los_rios.csv',APOYO/'Fuentes/AnalisisK.shp',
         APOYO/'Fuentes/terrazas_170_UTM18S_con_edad_referencia_Jara_Munoz_2015.csv',KP_MANUALES,DEM,
         REVISION_B/'Decisiones_B_20261019.csv',REVISION_B/'Tres_PT_todos_los_rios_20261019.csv',
         REVISION_B/'K_oficiales_tres_PT_20261019.csv']
trace={'fecha_utc':datetime.now(timezone.utc).isoformat(),'definicion_H':'DEM en XY de la terraza J-M más cercana del mismo MIS',
       'parametros':{'theta':THETA,'A0':A0,'n':N,'nivel_mar':NIVEL_MAR},'origen_geometria':origen_geometria,
       'n_K':len(resultados),'n_B':len(casos),'n_rios':int(resultados.rid.nunique()),
       'roles':seleccion_todos.groupby('rol_resultado').size().astype(int).to_dict(),
       'control_revision':control_revision,'control_tau':control_tau,
       'controles_tau_otros':{k:v['control_tau'] for k,v in productos_otros.items()},
       'max_error_H_PT_m':float(max(errores)),'equivalencia_U_aparente':True,
       'cronologia_publicada':pub.POLITICA,'seleccion_B_oficial':True,'K_unico_adoptado':False,'promedios_entre_MIS':False,
       'n_posiciones_PT_oficiales_distintas_por_caso':int(resultados.groupby('caso_B').s_PT_m.nunique().sum()),
       'visibilidad':{'figuras':MOSTRAR,'K':MOSTRAR_K,'comparaciones':SERIES},
       'fuentes_sha256':{str(p):sha(p) for p in sources}}
(SALIDA/'Trazabilidad.json').write_text(json.dumps(trace,ensure_ascii=False,indent=2),encoding='utf8')
from html import escape
page=['<!doctype html><meta charset="utf-8"><title>K, U y τ oficiales</title><style>body{font:16px system-ui;max-width:1450px;margin:30px auto;padding:20px;color:#19394b}img{width:100%;max-width:1350px}</style><h1>K, U y τ · calibraciones oficiales</h1>',
      '<p>Ocho casos B/MIS oficiales; tres etiquetas P_T por caso. No son 24 mediciones independientes ni se adopta un K promedio.</p>',
      '<p><a href="Norte_sur/Comparacion_K_tau_U_norte_sur.html">Comparación regional oficial</a> · <a href="K_Grant/ABRIR_K_TERRAZA_DEM.html">K oficiales por MIS</a> · <a href="K_oficiales_tres_PT.csv">Tabla oficial</a></p>',
      '<p>U aparente = dz/dτ = K×ksn; su interpretación como uplift requiere equilibrio. Los rangos son escenarios, no intervalos de confianza.</p>']
for f in [*sorted((SALIDA/'Figuras_por_rio').glob('*.png')),*tau_fig,*figuras_NS,tabla]:page.append('<img src="'+escape(f.relative_to(SALIDA).as_posix())+'">')
for rol,carpeta in [('comparacion','Comparacion'),('prueba','Prueba')]:
    if rol in productos_otros:page.append('<h2>'+carpeta+'</h2><p><a href="'+carpeta+'/K_Grant/ABRIR_K_TERRAZA_DEM.html">Figuras K</a> · <a href="'+carpeta+'/K_tres_PT.csv">Tabla K</a> · <a href="'+carpeta+'/Tau/Tau_KP_por_escenario.csv">τ por escenario</a></p>')
(SALIDA/'ABRIR_RESULTADOS.html').write_text('\n'.join(page),encoding='utf8')
marker={'salida':str(SALIDA),'notebook':str(BASE/'Notebooks_principales_20261019/03_K_U_tau_20261019.ipynb'),
        'alcance':'oficiales; comparación y prueba en carpetas separadas','n_casos_oficiales':len(casos),'n_escenarios_oficiales':len(resultados)}
temporary=APOYO/'Ultima_corrida_03.tmp'
temporary.write_text(json.dumps(marker,ensure_ascii=False,indent=2),encoding='utf8')
temporary.replace(APOYO/'Ultima_corrida_03.json')
enlace('ABRIR RESULTADOS OFICIALES Y SECCIONES DE COMPARACIÓN/PRUEBA',SALIDA/'ABRIR_RESULTADOS.html')
print('Verificado: revisión reproducida; 8 casos oficiales y 24 etiquetas P_T, con fuentes y roles trazables.')

## 12. Qué quedó en otros notebooks
- **01:** hidrología, extracción original y catálogo de perfiles.
- **02:** morfometría descriptiva, KP, litología, sensibilidad de θ y mapas de los 18 ríos. No calibra K.
- **04:** ksn de los cauces seleccionados y comparación por rumbo de falla. Sus ajustes son diagnósticos y no alimentan silenciosamente esta calibración.
- **05:** red completa del polígono, ksn en ventana de 1000 m y perfiles χ con litología de los cauces que cruzan fallas.

Las variantes antiguas K con B=0 y calibraciones regionales transferidas se conservan en el respaldo y sus productos, pero no sustituyen los K con B manual. J-M publica U de terrazas; los K derivados de U J-M se calculan aquí y llevan ese rótulo.

Referencias: [Perron y Royden, 2013](https://doi.org/10.1002/esp.3302); [Royden y Perron, 2013](https://doi.org/10.1002/jgrf.20031); [Jara-Muñoz et al., 2015](https://doi.org/10.1016/j.quascirev.2015.01.005).

La selección vigente está guardada en la carpeta Revision_K_por_rio_20261019. El módulo integracion_K_oficial aplica roles y tres P_T antes de alimentar K, U y τ principales. Cambiar una figura no modifica las decisiones.